In [1]:
import sys
import time
import random
import numpy as np
import pandas as pd
from functools import partial
import matplotlib.pyplot as plt
import pennylane as qp

## How to Calculate Fidelity between 2 QROMs?
Fidelity is often computed with no noise at all to determine how logically equivalent two circuits are.

Noise-added fidelity is what people often do for hardware studies. The fidelity drop is caused by:

depolarizing noise
amplitude damping
readout errors
etc.

This measures hardware robustness.

The "ideal QROM" is not a particular circuit architecture.

It is the desired unitary transformation:

$∣a⟩∣0⟩→∣a⟩∣d(a)⟩$

where d(a) is the stored data.

In [ ]:
def evaluate_QROM(bitstrings, num_index_bits, num_swap_bits=0, bits_per_value=2, print_out=False):

    if(len(bitstrings) != 2**num_index_bits or len(bitstrings[0]) != bits_per_value):
        print(f"Error: Mismatch in size of bitstrings and expected number of num_index_bits and bits_per_value. ({len(bitstrings)}, {len(bitstrings[0])}) != ({2**num_index_bits}, {bits_per_value})")
        return -1, -1, -1, -1
    
    control_wires = np.arange(num_index_bits)
    target_wires  = np.arange(num_index_bits, num_index_bits + bits_per_value)
    work_wires    = np.arange(target_wires[-1] + 1, target_wires[-1] + 1 + bits_per_value*(2**num_swap_bits - 1))
    
    @partial(qp.compile, basis_set="CNOT")  # Line added for resource estimation purposes only.
    @qp.set_shots(1)
    @qp.qnode(qp.device("default.qubit"))
    def circuit(index):
        qp.BasisState(index, wires=control_wires)
        qp.QROM(bitstrings, control_wires, target_wires, work_wires, clean=False)
        return qp.sample(wires=target_wires), qp.sample(wires=target_wires)
    
    r = qp.specs(circuit)(0)["resources"]

    num_qubits = len(control_wires) + len(target_wires) + len(work_wires)
    oneQ = r.gate_sizes.get(1,0)
    twoQ = r.gate_sizes.get(2,0)

    if print_out:
        print("Number of qubits: ", num_qubits)
        print("One-qubit gates: ", oneQ)
        print("Two-qubit gates: ", twoQ)
        print(f"Depth: {r.depth}")

    return {
        "swap_bits": num_swap_bits,
        "qubits": num_qubits,
        "oneQ": oneQ,
        "twoQ": twoQ,
        "depth": r.depth,
    }

In [2]:
def generate_bitstrings(array_length, bit_length, worstcase=False):
    if worstcase: return [f"{2**bit_length - 1:0{bit_length}b}" for _ in range(array_length)]
    return [f"{random.randint(0, 2**bit_length - 1):0{bit_length}b}" for _ in range(array_length)]

Ok, just make it into the truth table of the ideal memory elements. And we're good ig

In [ ]:
# Psi is the state vector
psi_discovered = discovered_qrom(address)
psi_ideal      = ideal_qrom(address)

# Rho is the state vector * state vector^T or something, and it becomes a density matrix
rho_discovered = qml.math.dm_from_state_vector(psi_discovered)
rho_ideal      = qml.math.dm_from_state_vector(psi_ideal)

F = qml.math.fidelity(rho_discovered, rho_ideal)